# HS524M Test 2 – Q1: Interactive Web Map of Waste-Bin Observations (IIT Tirupati)

**Question:** Where are the waste bins on campus, and what do they look like?

**Workflow (same step-by-step style as the lectures):** read the QField data → inspect it → clean it → prepare photos for the web → build a Folium map → save HTML → check it before hosting.

| Item | Choice |
|---|---|
| Field data | QField `notes.gpkg` (point observations: title, description, photo) |
| Photos | `DCIM/` folder from the QField project |
| Library | Folium (Leaflet in Python) |
| Output | `waste_bins_map.html` at the repo root + resized photos in `assets/waste_bins/photos/` |

Run from the root of the GitHub repo.

## 1. Libraries and folders

In [1]:
import os
import re
import html
import base64

import geopandas as gpd
import pyogrio
import pandas as pd
import folium
from folium.plugins import Fullscreen, MiniMap, MeasureControl, MousePosition
from PIL import Image, ImageOps

os.makedirs("assets/waste_bins/photos", exist_ok=True)
print("Libraries loaded and output folders ready.")

Libraries loaded and output folders ready.


## 2. Paths and settings

All the settings are in this one cell. The photos are **copied and resized** into the repo, so they still show when the map is hosted online. A path like `C:\Users\...` would only work on your own computer.

In [2]:
project_folder = r"C:\Users\shara\Downloads\Waste_Bins_on_Campus_IIT_Tirupati"
notes_file  = project_folder + r"\notes.gpkg"
tracks_file = project_folder + r"\tracks.gpkg"

MY_NAME = "Sharayu Dahibhate"

# Optional: observations from classmates (must be acknowledged in the popup).
# Each entry needs the collector's name, their GeoPackage and the folder holding their photos.
other_students = [
    # {"name": "Student Name", "gpkg": r"C:\path\to\their\notes.gpkg", "photos": r"C:\path\to\their\DCIM"},
]

# Layers inside notes.gpkg. For the other columns, leave None to detect automatically.
NOTES_LAYER = "notes_point"            # the note locations
ATTACH_LAYER = "notes_attachments"    # the photo records, linked to notes by note_uuid
TITLE_COL = None
DESC_COL = None
PHOTO_COL = None

EMBED_PHOTOS_IN_HTML = False   # True = photos stored inside the HTML (bigger file, but cannot break)

## 3. Inspect the QField data

Check the layers inside the GeoPackage, the columns, the CRS and the number of observations before doing anything else.

In [3]:
import pyogrio
print(pyogrio.list_layers(notes_file))

[['notes_point' 'Point Z']
 ['notes_attachments' None]]


In [4]:
notes_raw = gpd.read_file(notes_file, layer=NOTES_LAYER) if NOTES_LAYER else gpd.read_file(notes_file)

print("Observations:", len(notes_raw))
print("CRS:", notes_raw.crs)
print("Geometry types:", notes_raw.geom_type.value_counts().to_dict())
print("Columns:", notes_raw.columns.tolist())
print("\nPhoto records (attachments layer):")
attachments_raw = pd.DataFrame(gpd.read_file(notes_file, layer=ATTACH_LAYER)) if ATTACH_LAYER else None
print(len(attachments_raw), "rows |", attachments_raw.columns.tolist())
notes_raw.drop(columns="geometry").head()

Observations: 60
CRS: EPSG:4326
Geometry types: {'Point': 60}
Columns: ['uuid', 'color', 'title', 'note', 'timestamp', 'geometry']

Photo records (attachments layer):
58 rows | ['note_layer', 'note_uuid', 'media', 'description', 'timestamp']


,uuid,color,title,note,timestamp
0,{3f86f16f-7a00-4866-96f3-65dc5ccd434f},#377eb8,Ab2 floor 1,Dry waste bin,2026-09-06 11:17:32.552000+00:00
1,{5052545a-6326-4bd6-a866-5fabc473f4b7},#377eb8,Dry waste bin,Ab2 floor 1,2026-09-06 11:18:43.778000+00:00
2,{7774d828-6660-4bdd-9ea0-313ddf29ffa4},#377eb8,Dry waste bin,Ab2 floor 0,2026-09-06 11:19:54.335000+00:00
3,{ab75e3c5-5153-4570-957f-56305567ec9d},#377eb8,Dry waste bin,Ab2 floor 0,2026-09-06 11:21:07.262000+00:00
4,{43b35555-3a68-4830-8370-d49757175bd1},#377eb8,Dry waste bin,Outside ab1 library,2026-09-06 11:22:56.727000+00:00


## 4. Function to turn QField data into map-ready observations

QField keeps the notes and their photographs in **two linked tables** inside `notes.gpkg`:
- `notes_point`: one row per note (location, title, note text, timestamp, `uuid`),
- `notes_attachments`: one row per photograph (`media` = file path, `note_uuid` = the note it belongs to).

For each GeoPackage the function:
1. reads the notes, drops empty geometries and converts to WGS84 (EPSG:4326, which Folium uses),
2. reads the attachments and joins each photo to its note through `note_uuid` = `uuid`,
3. finds each photo file in the DCIM folder, copies it to `assets/waste_bins/photos/` after fixing its rotation and resizing it to at most 900 px,
4. keeps the other attributes (timestamp in IST) for the popup,
5. records who collected the observation.

A note can have several photos or none, so every popup shows all of its photos.

In [5]:
def pick(columns, candidates):
    lower = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None

def has_value(v):
    return not pd.isna(v) and str(v).strip() not in ("", "None", "nan", "NaT")

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")

def norm_id(x):
    return str(x).strip().strip("{}").lower()

def fmt_value(v):
    if isinstance(v, pd.Timestamp):
        if v.tzinfo is not None:
            return v.tz_convert("Asia/Kolkata").strftime("%Y-%m-%d %H:%M") + " IST"
        return v.strftime("%Y-%m-%d %H:%M")
    return v

def build_photo_index(folder):
    index = {}
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                index[f.lower()] = os.path.join(root, f)
    return index

def find_photo_file(value, index):
    """Match a stored path/filename to a real file in DCIM (relative path, Android path,
    bare filename, other extension case, missing extension, or the timestamp number)."""
    if not has_value(value):
        return None
    text = str(value).strip().strip("'\"").replace("\\", "/")
    base = os.path.basename(text).lower()
    if base in index:
        return index[base]
    stem = os.path.splitext(base)[0]
    for name, path in index.items():
        if os.path.splitext(name)[0] == stem:
            return path
    for digits in re.findall(r"\d{12,}", text):
        for name, path in index.items():
            if digits in name:
                return path
    return None

def prepare_photo(value, index, prefix=""):
    src = find_photo_file(value, index)
    if src is None:
        return None
    name = re.sub(r"[^a-z0-9._-]", "_", prefix + os.path.basename(src).lower())
    name = os.path.splitext(name)[0] + ".jpg"            # lowercase names: GitHub Pages is case-sensitive
    out_path = "assets/waste_bins/photos/" + name
    if not os.path.exists(out_path):
        with Image.open(src) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")   # fix phone-photo rotation
            im.thumbnail((900, 900))
            im.save(out_path, "JPEG", quality=80, optimize=True)
    return out_path

def load_observations(gpkg_path, photo_folder, collector, layer=None, attach_layer=None, prefix=""):
    layers = [l[0] for l in pyogrio.list_layers(gpkg_path)]
    layer = layer or next((l for l in layers if "attach" not in l.lower()), layers[0])
    attach_layer = attach_layer or next((l for l in layers if "attach" in l.lower()), None)
    print(f"[{collector}] notes layer: {layer} | attachments layer: {attach_layer}")

    gdf = gpd.read_file(gpkg_path, layer=layer)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].to_crs(epsg=4326).copy()
    gdf["geometry"] = gdf.geometry.representative_point()

    index = build_photo_index(photo_folder)
    print(f"[{collector}] image files found in photo folder: {len(index)}")

    cols = [c for c in gdf.columns if c != "geometry"]
    title_col = TITLE_COL or pick(cols, ["title", "name", "location", "bin_name", "label"])
    desc_col  = DESC_COL  or pick(cols, ["description", "desc", "note", "notes", "comment", "comments",
                                         "remarks", "details", "field_note"])
    uuid_col  = pick(cols, ["uuid"])
    print(f"[{collector}] title: {title_col} | description: {desc_col} | link id: {uuid_col}")

    # QField stores photos in a separate attachments table, linked to the note by note_uuid
    photos_by_note = {}
    if attach_layer:
        att = pd.DataFrame(gpd.read_file(gpkg_path, layer=attach_layer))
        key_col   = pick(att.columns, ["note_uuid", "uuid", "parent_uuid"])
        media_col = pick(att.columns, ["media", "photo", "path", "file", "filename", "attachment"])
        time_col  = pick(att.columns, ["timestamp"])
        print(f"[{collector}] attachment link column: {key_col} | photo column: {media_col} | rows: {len(att)}")
        if time_col:
            att = att.sort_values(time_col)
        for _, a in att.iterrows():
            photo = prepare_photo(a[media_col], index, prefix) if media_col else None
            if photo and key_col:
                photos_by_note.setdefault(norm_id(a[key_col]), []).append(photo)

    skip = {title_col, desc_col, "fid", "uuid", "color", "geometry"}

    rows = []
    for i, (_, r) in enumerate(gdf.iterrows(), start=1):
        photos = photos_by_note.get(norm_id(r[uuid_col]), []) if uuid_col else []
        rows.append({
            "title": str(r[title_col]) if title_col and has_value(r[title_col]) else f"Waste bin {i}",
            "description": str(r[desc_col]) if desc_col and has_value(r[desc_col]) else "",
            "collector": collector,
            "lat": r.geometry.y,
            "lon": r.geometry.x,
            "photos": photos,
            "photo_web": photos[0] if photos else None,
            "n_photos": len(photos),
            "extra": {c: fmt_value(r[c]) for c in cols if c not in skip and has_value(r[c])},
        })
    out = pd.DataFrame(rows)
    print(f"[{collector}] observations: {len(out)} | with photo: {(out['n_photos'] > 0).sum()} | photos in total: {out['n_photos'].sum()}")
    return out

## 5. Load my observations (and classmates', if any)

In [6]:
obs_me = load_observations(notes_file, project_folder, MY_NAME, layer=NOTES_LAYER, attach_layer=ATTACH_LAYER)
obs_me.drop(columns="extra").head()

[Sharayu Dahibhate] notes layer: notes_point | attachments layer: notes_attachments
[Sharayu Dahibhate] image files found in photo folder: 61
[Sharayu Dahibhate] title: title | description: note | link id: uuid
[Sharayu Dahibhate] attachment link column: note_uuid | photo column: media | rows: 58
[Sharayu Dahibhate] observations: 60 | with photo: 58 | photos in total: 58


,title,description,collector,lat,lon,photos,photo_web,n_photos
0,Ab2 floor 1,Dry waste bin,Sharayu Dahibhate,13.716438,79.591198,[assets/waste_bins/photos/jpeg_202609061117432...,assets/waste_bins/photos/jpeg_2026090611174323...,1
1,Dry waste bin,Ab2 floor 1,Sharayu Dahibhate,13.716331,79.591418,[assets/waste_bins/photos/jpeg_202609061118526...,assets/waste_bins/photos/jpeg_2026090611185265...,1
2,Dry waste bin,Ab2 floor 0,Sharayu Dahibhate,13.716331,79.591418,[assets/waste_bins/photos/jpeg_202609061120009...,assets/waste_bins/photos/jpeg_2026090611200096...,1
3,Dry waste bin,Ab2 floor 0,Sharayu Dahibhate,13.716331,79.591418,[assets/waste_bins/photos/jpeg_202609061121151...,assets/waste_bins/photos/jpeg_2026090611211513...,1
4,Dry waste bin,Outside ab1 library,Sharayu Dahibhate,13.716331,79.591418,[assets/waste_bins/photos/jpeg_202609061123024...,assets/waste_bins/photos/jpeg_2026090611230244...,1


In [7]:
obs_others = []
for s in other_students:
    obs_others.append(
        load_observations(s["gpkg"], s["photos"], s["name"], prefix=slug(s["name"]) + "_")
    )

all_obs = pd.concat([obs_me] + obs_others, ignore_index=True)
print("Total observations on the map:", len(all_obs))

missing = all_obs[all_obs["n_photos"] == 0]
print("Observations without a photo:", len(missing))
missing[["title", "collector"]]

Total observations on the map: 60
Observations without a photo: 2


,title,collector
43,Dry waste bin,Sharayu Dahibhate
49,Dry waste bin,Sharayu Dahibhate


## 6. Popup content

Each popup shows the title, the collector (clearly), the description, the coordinates, every other recorded attribute, and the photograph. Classmates' data is labelled as acknowledged.

In [8]:
def photo_src(path):
    if not EMBED_PHOTOS_IN_HTML:
        return path
    with open(path, "rb") as f:
        return "data:image/jpeg;base64," + base64.b64encode(f.read()).decode()

def popup_html(r):
    extra_rows = "".join(
        f"<tr><td style='padding:2px 8px 2px 0;color:#666'>{html.escape(str(k))}</td>"
        f"<td style='padding:2px 0'>{html.escape(str(v))}</td></tr>"
        for k, v in r["extra"].items()
    )
    source_note = "" if r["collector"] == MY_NAME else (
        "<div style='font-size:11px;color:#a05a00'>Data acknowledged from classmate</div>"
    )
    desc = f"<p style='margin:6px 0'>{html.escape(r['description'])}</p>" if r["description"] else ""
    if r["photos"]:
        imgs = "".join(
            f"<a href='{photo_src(p)}' target='_blank'>"
            f"<img src='{photo_src(p)}' alt='Waste bin photo' style='width:100%;border-radius:6px;margin-top:6px'></a>"
            for p in r["photos"]
        )
        count = (f"<div style='font-size:11px;color:#666'>{len(r['photos'])} photos - scroll</div>"
                 if len(r["photos"]) > 1 else "")
        photo = f"<div style='max-height:300px;overflow-y:auto'>{imgs}</div>{count}"
    else:
        photo = "<i style='color:#888'>No photo recorded</i>"
    return (
        "<div style='font-family:system-ui,sans-serif;font-size:13px;width:280px'>"
        f"<div style='font-size:15px;font-weight:600'>{html.escape(r['title'])}</div>"
        f"<div>Collected by: <b>{html.escape(r['collector'])}</b></div>{source_note}"
        f"{desc}"
        f"<table style='font-size:12px'>{extra_rows}"
        f"<tr><td style='padding:2px 8px 2px 0;color:#666'>Coordinates</td>"
        f"<td>{r['lat']:.5f}, {r['lon']:.5f}</td></tr></table>"
        f"{photo}</div>"
    )

## 7. Build the Folium map

- Three basemaps (OpenStreetMap, light map, Esri satellite) switchable in the Layer Control.
- One layer per collector, each with its own marker colour.
- Optional GPS survey track layer from `tracks.gpkg` (hidden by default).
- Controls: Layer Control, fullscreen, mini-map, measure tool, mouse coordinates, scale bar.

In [9]:
m = folium.Map(
    location=[all_obs["lat"].mean(), all_obs["lon"].mean()],
    zoom_start=17,
    tiles=None,
    control_scale=True,
)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    name="Light map",
    show=False,
).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics",
    name="Satellite (Esri)",
    show=False,
).add_to(m)

colors = ["darkblue", "orange", "green", "purple", "red", "cadetblue", "darkred", "pink"]
legend_items = []

for k, (collector, grp) in enumerate(all_obs.groupby("collector", sort=False)):
    color = colors[k % len(colors)]
    fg = folium.FeatureGroup(name=f"Waste bins – {collector} ({len(grp)})")
    for _, r in grp.iterrows():
        folium.Marker(
            location=[r["lat"], r["lon"]],
            tooltip=f"{r['title']} — {r['collector']}",
            popup=folium.Popup(folium.Html(popup_html(r), script=True), max_width=320),
            icon=folium.Icon(color=color, icon="trash", prefix="fa"),
        ).add_to(fg)
    fg.add_to(m)
    legend_items.append((collector, color, len(grp)))

# GPS survey tracks recorded by QField (optional)
try:
    tracks = gpd.read_file(tracks_file).to_crs(epsg=4326)
    folium.GeoJson(
        tracks[["geometry"]],
        name="Survey tracks (GPS)",
        style_function=lambda f: {"color": "#d62728", "weight": 3},
        show=False,
    ).add_to(m)
except Exception as err:
    print("Tracks layer skipped:", err)

m.fit_bounds([[all_obs["lat"].min(), all_obs["lon"].min()],
              [all_obs["lat"].max(), all_obs["lon"].max()]], padding=(30, 30))

Fullscreen().add_to(m)
MiniMap(toggle_display=True, position="bottomleft").add_to(m)
MeasureControl(primary_length_unit="meters").add_to(m)
MousePosition(position="bottomright", num_digits=5).add_to(m)

# Title and legend
swatch = {"darkblue": "#0b3c8c", "orange": "#f69730", "green": "#72b026", "purple": "#d252b9",
          "red": "#d33d29", "cadetblue": "#436978", "darkred": "#a23336", "pink": "#ff8ee9"}
legend_rows = "".join(
    f"<div><span style='display:inline-block;width:12px;height:12px;border-radius:50%;"
    f"background:{swatch.get(c, '#666')};margin-right:6px'></span>{html.escape(n)} ({cnt})</div>"
    for n, c, cnt in legend_items
)
m.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:9999;"
    "background:white;padding:6px 14px;border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.3);"
    "font:600 14px system-ui,sans-serif'>Waste bins on campus – IIT Tirupati</div>"
    "<div style='position:fixed;bottom:30px;right:10px;z-index:9999;background:white;padding:8px 12px;"
    "border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.3);font:12px system-ui,sans-serif'>"
    "<b>Collected by</b>" + legend_rows + "</div>"
))

folium.LayerControl(collapsed=False).add_to(m)

m.save("waste_bins_map.html")
m

C:\Users\shara\anaconda3\envs\uisa_2026\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(


## 8. Check before hosting

The hosted map must not contain any path that exists only on your computer.

In [10]:
with open("waste_bins_map.html", encoding="utf-8") as f:
    page = f.read()

bad = [p for p in ["C:" + chr(92), "C:/", "file:///", "/storage/", "Downloads"] if p in page]
print("Local-path problems found:", bad if bad else "none")

photos = [p for lst in all_obs["photos"] for p in lst]
print("Photos referenced:", len(photos), "| all exist:", all(os.path.exists(p) for p in photos))
print("Total photo size (MB):", round(sum(os.path.getsize(p) for p in photos) / 1e6, 1))

Local-path problems found: none
Photos referenced: 58 | all exist: True
Total photo size (MB): 6.8


## 9. Tables and numbers for the webpage

In [11]:
table = all_obs.drop(columns=["extra", "photos"]).assign(photo_files=all_obs["photos"].apply(" | ".join))
table.round(6).to_csv("assets/waste_bins/bins_table.csv", index=False)

summary = (all_obs.groupby("collector")
           .agg(observations=("title", "count"), with_photo=("n_photos", lambda s: (s > 0).sum()), photos=("n_photos", "sum"))
           .reset_index())
summary.to_csv("assets/waste_bins/summary.csv", index=False)
summary

,collector,observations,with_photo,photos
0,Sharayu Dahibhate,60,58,58
